Tech Challenge Fase 2  
## Notebook 06 — Monitoring e Observabilidade

### Responsabilidade do notebook

Este notebook consolida informações operacionais e técnicas do pipeline de dados.

O objetivo é permitir o acompanhamento de:

- disponibilidade das camadas Bronze, Silver e Gold;
- volume de arquivos;
- volume armazenado;
- quantidade de registros;
- regras de qualidade;
- registros inválidos;
- datasets aprovados, em atenção ou reprovados;
- disponibilidade dos produtos para Power BI e Machine Learning;
- integridade da pipeline Streaming;
- status geral da plataforma.

---

### Entradas

```text
config/config.json
config/bronze_metadata
config/silver_metadata
config/gold_metadata
logs/data_quality/dashboard/
streaming/
```

### Saídas

```text
logs/monitoring/pipeline_inventory
logs/monitoring/storage_metrics
logs/monitoring/data_quality_metrics
logs/monitoring/platform_status
logs/monitoring/history
gold/exports_powerbi/monitoring_dashboard
```

## 1. Contexto do Monitoring

O Monitoring é uma camada transversal.

```text
Bronze
  ↓
Silver
  ↓
Gold
  ↓
Power BI / IA

↕ Monitoring
```

A observabilidade permite detectar:

- arquivos ausentes;
- crescimento inesperado de volume;
- redução no número de registros;
- aumento de rejeitados;
- reprovação de regras críticas;
- falha na atualização de produtos Gold;
- indisponibilidade da pipeline Streaming.

## 2. Importação das bibliotecas

In [0]:
import json
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    BooleanType
)

## 3. Leitura das configurações oficiais

Todos os caminhos são obtidos do `config.json`.

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    dbutils.fs.head(CONFIG_FILE_PATH)
)

BASE_PATH = config["environment"]["base_path"]
BRONZE_PATH = config["paths"]["bronze_path"]
SILVER_PATH = config["paths"]["silver_path"]
GOLD_PATH = config["paths"]["gold_path"]
STREAMING_PATH = config["paths"]["streaming_path"]
LOG_PATH = config["paths"]["log_path"]
CONFIG_PATH = config["paths"]["config_path"]
EXECUTION_DATE = config["project"]["execution_date"]

MONITORING_ROOT_PATH = f"{LOG_PATH}/monitoring"
MONITORING_INVENTORY_PATH = (
    f"{MONITORING_ROOT_PATH}/pipeline_inventory"
)
MONITORING_STORAGE_PATH = (
    f"{MONITORING_ROOT_PATH}/storage_metrics"
)
MONITORING_QUALITY_PATH = (
    f"{MONITORING_ROOT_PATH}/data_quality_metrics"
)
MONITORING_STATUS_PATH = (
    f"{MONITORING_ROOT_PATH}/platform_status"
)
MONITORING_HISTORY_PATH = (
    f"{MONITORING_ROOT_PATH}/history"
)

QUALITY_DASHBOARD_PATH = (
    f"{LOG_PATH}/data_quality/dashboard"
)

POWERBI_MONITORING_PATH = (
    f"{GOLD_PATH}/exports_powerbi/"
    f"monitoring_dashboard"
)

print("MONITORING_ROOT_PATH:", MONITORING_ROOT_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 4. Criação dos diretórios de saída

In [0]:
for path in [
    MONITORING_ROOT_PATH,
    MONITORING_INVENTORY_PATH,
    MONITORING_STORAGE_PATH,
    MONITORING_QUALITY_PATH,
    MONITORING_STATUS_PATH,
    MONITORING_HISTORY_PATH,
    POWERBI_MONITORING_PATH
]:
    dbutils.fs.mkdirs(path)

print("Diretórios de Monitoring criados/validados.")

## 5. Funções auxiliares

As funções abaixo permitem:

- verificar caminhos;
- listar arquivos recursivamente;
- calcular volume armazenado;
- contar arquivos;
- estimar quantidade de registros;
- carregar metadados de forma segura.

In [0]:
def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def list_files_recursive(path: str) -> list:
    files = []

    try:
        for item in dbutils.fs.ls(path):

            if item.path.endswith("/"):
                files.extend(
                    list_files_recursive(
                        item.path
                    )
                )
            else:
                files.append(item)

    except Exception:
        pass

    return files


def directory_metrics(path: str) -> dict:
    files = list_files_recursive(path)

    total_files = len(files)

    total_bytes = sum(
        int(item.size)
        for item in files
    )

    return {
        "file_count": total_files,
        "size_bytes": total_bytes,
        "size_mb": round(
            total_bytes / 1024 / 1024,
            4
        ),
        "size_gb": round(
            total_bytes / 1024 / 1024 / 1024,
            6
        )
    }


def safe_read_parquet(path: str):
    if not path_exists(path):
        return None

    try:
        return spark.read.parquet(path)
    except Exception:
        return None


def safe_read_csv(path: str):
    if not path_exists(path):
        return None

    try:
        return (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .option("sep", ";")
            .option("encoding", "UTF-8")
            .csv(path)
        )
    except Exception:
        return None

## 6. Definição dos componentes monitorados

Nesta etapa registramos os principais componentes da plataforma.

Cada componente possui:

- camada;
- dataset;
- ano;
- caminho;
- formato;
- criticidade.

In [0]:
monitoring_targets = []

def add_target(
    layer,
    dataset,
    ano,
    path,
    file_format,
    critical=True
):
    monitoring_targets.append({
        "layer": layer,
        "dataset": dataset,
        "ano": int(ano),
        "path": path,
        "file_format": file_format,
        "critical": bool(critical)
    })


for dataset in [
    "alunos",
    "estados",
    "municipios",
    "metas_municipios",
    "metas_ufs"
]:
    for ano in [2023, 2024, 2025]:

        add_target(
            layer="bronze",
            dataset=dataset,
            ano=ano,
            path=(
                f"{BRONZE_PATH}/"
                f"{dataset}/ano={ano}"
            ),
            file_format="parquet",
            critical=True
        )

        add_target(
            layer="silver",
            dataset=dataset,
            ano=ano,
            path=(
                f"{SILVER_PATH}/"
                f"{dataset}/ano={ano}"
            ),
            file_format="csv",
            critical=True
        )


for dataset in [
    "alunos",
    "municipios",
    "estados"
]:
    for ano in [2023, 2024, 2025]:

        add_target(
            layer="gold",
            dataset=dataset,
            ano=ano,
            path=(
                f"{GOLD_PATH}/"
                f"{dataset}/ano={ano}"
            ),
            file_format="csv",
            critical=True
        )


add_target(
    layer="gold",
    dataset="indicadores",
    ano=0,
    path=(
        f"{GOLD_PATH}/indicadores"
    ),
    file_format="csv",
    critical=True
)

add_target(
    layer="gold",
    dataset="machine_learning",
    ano=0,
    path=(
        f"{GOLD_PATH}/base_modelo_ia"
    ),
    file_format="csv",
    critical=True
)

add_target(
    layer="gold",
    dataset="powerbi",
    ano=0,
    path=(
        f"{GOLD_PATH}/exports_powerbi"
    ),
    file_format="csv",
    critical=True
)


for dataset, path in [
    (
        "streaming_input",
        f"{STREAMING_PATH}/input"
    ),
    (
        "streaming_bronze",
        f"{STREAMING_PATH}/bronze_streaming"
    ),
    (
        "streaming_silver",
        f"{STREAMING_PATH}/silver_streaming"
    ),
    (
        "streaming_gold",
        f"{STREAMING_PATH}/gold_streaming/current"
    )
]:
    add_target(
        layer="streaming",
        dataset=dataset,
        ano=0,
        path=path,
        file_format="parquet",
        critical=False
    )

print(
    "Componentes monitorados:",
    len(monitoring_targets)
)

## 7. Inventário e disponibilidade da plataforma

Esta etapa verifica:

- existência do componente;
- quantidade de arquivos;
- volume armazenado;
- disponibilidade para consumo.

In [0]:
inventory_results = []

for target in monitoring_targets:

    exists = path_exists(
        target["path"]
    )

    metrics = (
        directory_metrics(
            target["path"]
        )
        if exists
        else {
            "file_count": 0,
            "size_bytes": 0,
            "size_mb": 0.0,
            "size_gb": 0.0
        }
    )

    if not exists:
        component_status = "INDISPONIVEL"

    elif metrics["file_count"] == 0:
        component_status = "VAZIO"

    else:
        component_status = "DISPONIVEL"

    inventory_results.append({
        "layer": target["layer"],
        "dataset": target["dataset"],
        "ano": int(target["ano"]),
        "path": target["path"],
        "file_format": target["file_format"],
        "critical": bool(target["critical"]),
        "path_exists": bool(exists),
        "file_count": int(
            metrics["file_count"]
        ),
        "size_bytes": int(
            metrics["size_bytes"]
        ),
        "size_mb": float(
            metrics["size_mb"]
        ),
        "size_gb": float(
            metrics["size_gb"]
        ),
        "component_status": component_status,
        "execution_date": str(
            EXECUTION_DATE
        ),
        "evaluated_at": (
            datetime.now()
            .isoformat()
        )
    })

print(
    "Componentes avaliados:",
    len(inventory_results)
)

## 8. Criação do inventário monitorado

In [0]:
schema_inventory = StructType([
    StructField("layer", StringType(), False),
    StructField("dataset", StringType(), False),
    StructField("ano", IntegerType(), False),
    StructField("path", StringType(), False),
    StructField("file_format", StringType(), False),
    StructField("critical", BooleanType(), False),
    StructField("path_exists", BooleanType(), False),
    StructField("file_count", LongType(), False),
    StructField("size_bytes", LongType(), False),
    StructField("size_mb", DoubleType(), False),
    StructField("size_gb", DoubleType(), False),
    StructField("component_status", StringType(), False),
    StructField("execution_date", StringType(), False),
    StructField("evaluated_at", StringType(), False)
])

df_pipeline_inventory = (
    spark.createDataFrame(
        inventory_results,
        schema=schema_inventory
    )
)

display(
    df_pipeline_inventory
    .orderBy(
        "layer",
        "dataset",
        "ano"
    )
)

## 9. Métricas de armazenamento por camada

Esta visão consolida:

- quantidade de componentes;
- quantidade de arquivos;
- volume total;
- componentes indisponíveis.

In [0]:
df_storage_metrics = (
    df_pipeline_inventory
    .groupBy("layer")
    .agg(
        F.count("*").alias(
            "component_count"
        ),
        F.sum("file_count").alias(
            "file_count"
        ),
        F.sum("size_bytes").alias(
            "size_bytes"
        ),
        F.round(
            F.sum("size_mb"),
            4
        ).alias(
            "size_mb"
        ),
        F.round(
            F.sum("size_gb"),
            6
        ).alias(
            "size_gb"
        ),
        F.sum(
            F.when(
                F.col("component_status")
                != "DISPONIVEL",
                1
            ).otherwise(0)
        ).alias(
            "unavailable_components"
        )
    )
    .withColumn(
        "execution_date",
        F.lit(EXECUTION_DATE)
    )
)

display(
    df_storage_metrics
    .orderBy("layer")
)

## 10. Leitura das métricas de qualidade

O Monitoring reutiliza os resultados consolidados pelo `05_4_quality_dashboard`.

In [0]:
quality_kpis_path = (
    f"{QUALITY_DASHBOARD_PATH}/"
    f"kpis/execution_date={EXECUTION_DATE}"
)

quality_layer_path = (
    f"{QUALITY_DASHBOARD_PATH}/"
    f"layer_status/execution_date={EXECUTION_DATE}"
)

quality_ranking_path = (
    f"{QUALITY_DASHBOARD_PATH}/"
    f"dataset_ranking/execution_date={EXECUTION_DATE}"
)

df_quality_kpis = safe_read_parquet(
    quality_kpis_path
)

df_quality_layer_status = safe_read_parquet(
    quality_layer_path
)

df_quality_dataset_ranking = safe_read_parquet(
    quality_ranking_path
)

if df_quality_kpis is None:
    raise Exception(
        "KPIs de qualidade não encontrados. "
        "Execute o notebook 05_4_quality_dashboard."
    )

display(df_quality_kpis)

## 11. Construção do status geral da plataforma

O status geral considera:

- componentes críticos indisponíveis;
- regras bloqueantes reprovadas;
- regras não bloqueantes reprovadas;
- score de qualidade.

In [0]:
critical_components_unavailable = (
    df_pipeline_inventory
    .filter(
        F.col("critical")
        & (
            F.col("component_status")
            != "DISPONIVEL"
        )
    )
    .count()
)

quality_row = (
    df_quality_kpis
    .first()
)

blocking_failed_rules = int(
    quality_row[
        "blocking_failed_rules"
    ]
    or 0
)

failed_rules = int(
    quality_row[
        "failed_rules"
    ]
    or 0
)

quality_score = float(
    quality_row[
        "quality_score_percent"
    ]
    or 0.0
)

if (
    critical_components_unavailable > 0
    or blocking_failed_rules > 0
):
    platform_status = "CRITICO"

elif (
    failed_rules > 0
    or quality_score < 95.0
):
    platform_status = "ATENCAO"

else:
    platform_status = "SAUDAVEL"


platform_status_data = [{
    "execution_date": str(
        EXECUTION_DATE
    ),
    "platform_status": platform_status,
    "critical_components_unavailable": int(
        critical_components_unavailable
    ),
    "blocking_failed_rules": int(
        blocking_failed_rules
    ),
    "failed_rules": int(
        failed_rules
    ),
    "quality_score_percent": float(
        quality_score
    ),
    "evaluated_at": (
        datetime.now()
        .isoformat()
    )
}]

## 12. Criação do status da plataforma

In [0]:
schema_platform_status = StructType([
    StructField(
        "execution_date",
        StringType(),
        False
    ),
    StructField(
        "platform_status",
        StringType(),
        False
    ),
    StructField(
        "critical_components_unavailable",
        IntegerType(),
        False
    ),
    StructField(
        "blocking_failed_rules",
        IntegerType(),
        False
    ),
    StructField(
        "failed_rules",
        IntegerType(),
        False
    ),
    StructField(
        "quality_score_percent",
        DoubleType(),
        False
    ),
    StructField(
        "evaluated_at",
        StringType(),
        False
    )
])

df_platform_status = (
    spark.createDataFrame(
        platform_status_data,
        schema=schema_platform_status
    )
)

display(df_platform_status)

## 13. Identificação dos alertas operacionais

Esta visão apresenta componentes e regras que exigem atenção.

In [0]:
df_component_alerts = (
    df_pipeline_inventory
    .filter(
        F.col("component_status")
        != "DISPONIVEL"
    )
    .select(
        F.lit(
            "COMPONENT"
        ).alias(
            "alert_type"
        ),
        "layer",
        "dataset",
        "ano",
        F.col(
            "component_status"
        ).alias(
            "alert_status"
        ),
        F.col(
            "path"
        ).alias(
            "alert_message"
        ),
        "critical",
        "execution_date"
    )
)

quality_details_path = (
    f"{QUALITY_DASHBOARD_PATH}/"
    f"details/execution_date={EXECUTION_DATE}"
)

df_quality_details = safe_read_parquet(
    quality_details_path
)

if df_quality_details is None:

    df_quality_alerts = (
        df_component_alerts
        .limit(0)
    )

else:

    df_quality_alerts = (
        df_quality_details
        .filter(
            F.col("status")
            != "APROVADO"
        )
        .select(
            F.lit(
                "DATA_QUALITY"
            ).alias(
                "alert_type"
            ),
            "layer",
            "dataset",
            "ano",
            F.col(
                "status"
            ).alias(
                "alert_status"
            ),
            F.col(
                "message"
            ).alias(
                "alert_message"
            ),
            F.col(
                "blocking"
            ).alias(
                "critical"
            ),
            "execution_date"
        )
    )

df_monitoring_alerts = (
    df_component_alerts
    .unionByName(
        df_quality_alerts,
        allowMissingColumns=True
    )
)

if df_monitoring_alerts.count() == 0:
    print("Nenhum alerta operacional.")
else:
    display(
        df_monitoring_alerts
        .orderBy(
            F.col("critical").desc(),
            "layer",
            "dataset",
            "ano"
        )
    )

## 14. Persistência das métricas de Monitoring

As métricas serão persistidas por data de execução e também adicionadas ao histórico.

In [0]:
inventory_output_path = (
    f"{MONITORING_INVENTORY_PATH}/"
    f"execution_date={EXECUTION_DATE}"
)

storage_output_path = (
    f"{MONITORING_STORAGE_PATH}/"
    f"execution_date={EXECUTION_DATE}"
)

quality_output_path = (
    f"{MONITORING_QUALITY_PATH}/"
    f"execution_date={EXECUTION_DATE}"
)

status_output_path = (
    f"{MONITORING_STATUS_PATH}/"
    f"execution_date={EXECUTION_DATE}"
)

for df, path in [
    (
        df_pipeline_inventory,
        inventory_output_path
    ),
    (
        df_storage_metrics,
        storage_output_path
    ),
    (
        df_quality_kpis,
        quality_output_path
    ),
    (
        df_platform_status,
        status_output_path
    )
]:

    (
        df
        .coalesce(1)
        .write
        .mode("overwrite")
        .format("parquet")
        .option(
            "compression",
            "snappy"
        )
        .save(path)
    )

print("Métricas de Monitoring persistidas.")

## 15. Atualização do histórico

In [0]:
history_snapshot = (
    df_platform_status
    .withColumn(
        "snapshot_timestamp",
        F.current_timestamp()
    )
)

(
    history_snapshot
    .write
    .mode("append")
    .format("parquet")
    .partitionBy(
        "execution_date"
    )
    .save(
        MONITORING_HISTORY_PATH
    )
)

print(
    "Histórico atualizado em:",
    MONITORING_HISTORY_PATH
)

## 16. Exportação para Power BI

A exportação reúne inventário, armazenamento e status geral.

In [0]:
df_monitoring_powerbi = (
    df_pipeline_inventory
    .join(
        df_storage_metrics
        .select(
            "layer",
            F.col(
                "file_count"
            ).alias(
                "layer_file_count"
            ),
            F.col(
                "size_mb"
            ).alias(
                "layer_size_mb"
            ),
            F.col(
                "unavailable_components"
            )
        ),
        on="layer",
        how="left"
    )
    .crossJoin(
        df_platform_status
        .select(
            "platform_status",
            "quality_score_percent",
            "blocking_failed_rules",
            "critical_components_unavailable"
        )
    )
)

(
    df_monitoring_powerbi
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(
        POWERBI_MONITORING_PATH
    )
)

print(
    "Dashboard de Monitoring exportado para:",
    POWERBI_MONITORING_PATH
)

## 17. Checklist final

O notebook será concluído com sucesso quando:

- nenhum componente crítico estiver indisponível;
- nenhuma regra bloqueante estiver reprovada;
- o status da plataforma não for crítico.

In [0]:
if platform_status == "CRITICO":

    display(
        df_pipeline_inventory
        .filter(
            F.col("critical")
            & (
                F.col("component_status")
                != "DISPONIVEL"
            )
        )
    )

    raise Exception(
        "Monitoring identificou status "
        "CRITICO na plataforma."
    )

print(
    "Monitoring concluído com sucesso."
)

print(
    "Status da plataforma:",
    platform_status
)

print(
    "Score de qualidade:",
    quality_score
)

## Resultado esperado

Ao final deste notebook estarão disponíveis:

```text
logs/monitoring/pipeline_inventory/
logs/monitoring/storage_metrics/
logs/monitoring/data_quality_metrics/
logs/monitoring/platform_status/
logs/monitoring/history/

gold/exports_powerbi/monitoring_dashboard/
```

### Próxima etapa

```text
07_finops
```